# 🏠 Ames Housing — Linear Regression Model
**Proyek Machine Learning: Prediksi Harga Rumah**

Notebook ini mencakup:
1. Pembacaan data bersih (`train_clean.csv`) & pemisahan fitur/target
2. Pembagian data Training & Testing (80:20)
3. Mean Baseline sebagai tolok ukur awal
4. Pelatihan model Linear Regression
5. Evaluasi performa (MAE, MSE, RMSE, R² Score)
6. Perbandingan Baseline vs Linear Regression
7. Penyimpanan model ke file `.pkl`

---
## 1. Import Library & Membaca Dataset

In [ ]:
# Import library
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

print('✅ Library berhasil di-import!')

In [ ]:
# Membaca data bersih
df = pd.read_csv('train_clean.csv')

print(f'📊 Dataset berhasil dimuat!')
print(f'   Ukuran: {df.shape[0]} baris × {df.shape[1]} kolom')
print(f'   Kolom : {list(df.columns)}')
print()
df.head()

In [ ]:
# Memisahkan fitur (X) dan target (y)
feature_names = ['Gr Liv Area', 'Overall Qual', 'Garage Cars', 'Total Bsmt SF', 'Year Built']
target_name = 'SalePrice'

X = df[feature_names]
y = df[target_name]

print(f'🔢 Fitur (X): {X.shape}')
for f in feature_names:
    print(f'   • {f}')
print(f'\n🎯 Target (y): {y.shape}')
print(f'   • {target_name} — rata-rata: ${y.mean():,.0f}')

---
## 2. Pembagian Data — Training Set & Testing Set (80:20)

In [ ]:
# Split data 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f'📂 Pembagian Data:')
print(f'   Training Set : {X_train.shape[0]} sampel ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'   Testing Set  : {X_test.shape[0]} sampel ({X_test.shape[0]/len(X)*100:.0f}%)')
print(f'\n   Rata-rata SalePrice (Train) : ${y_train.mean():,.0f}')
print(f'   Rata-rata SalePrice (Test)  : ${y_test.mean():,.0f}')

---
## 3. Mean Baseline — Tolok Ukur Awal

In [ ]:
# Mean Baseline: prediksi semua data test dengan rata-rata SalePrice dari data training
baseline_pred_value = y_train.mean()
y_pred_baseline = np.full(shape=y_test.shape, fill_value=baseline_pred_value)

print(f'📏 Mean Baseline:')
print(f'   Prediksi konstan = ${baseline_pred_value:,.0f}')
print(f'   (Semua rumah di test set diprediksi seharga ${baseline_pred_value:,.0f})')

---
## 4. Melatih Model Linear Regression

In [ ]:
# Inisialisasi dan training model
model = LinearRegression()
model.fit(X_train, y_train)

print('✅ Model Linear Regression berhasil dilatih!')
print(f'\n📐 Koefisien Model:')
print(f'   {"Fitur":<20s} {"Koefisien":>15s}')
print(f'   {"-"*20} {"-"*15}')
for name, coef in zip(feature_names, model.coef_):
    print(f'   {name:<20s} {coef:>15.2f}')
print(f'\n   {"Intercept":<20s} {model.intercept_:>15.2f}')

In [ ]:
# Prediksi pada data testing
y_pred_lr = model.predict(X_test)

print(f'🔮 Prediksi pada Testing Set ({len(y_test)} sampel):')
print(f'   Contoh prediksi vs aktual (5 data pertama):')
print(f'   {"Aktual":>12s}  {"Prediksi":>12s}  {"Selisih":>12s}')
print(f'   {"-"*12}  {"-"*12}  {"-"*12}')
for actual, pred in zip(y_test.values[:5], y_pred_lr[:5]):
    diff = pred - actual
    print(f'   ${actual:>10,.0f}  ${pred:>10,.0f}  ${diff:>+10,.0f}')

---
## 5. Evaluasi Performa — Metrik MAE, MSE, RMSE, R²

In [ ]:
# Fungsi untuk menghitung semua metrik evaluasi
def evaluate_model(y_true, y_pred, model_name):
    """Menghitung MAE, MSE, RMSE, dan R² Score."""
    mae  = mean_absolute_error(y_true, y_pred)
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_true, y_pred)
    
    return {
        'Model': model_name,
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'R² Score': r2
    }

# Evaluasi Mean Baseline
metrics_baseline = evaluate_model(y_test, y_pred_baseline, 'Mean Baseline')

# Evaluasi Linear Regression
metrics_lr = evaluate_model(y_test, y_pred_lr, 'Linear Regression')

print('✅ Evaluasi selesai!')

---
## 6. Perbandingan Performa — Mean Baseline vs Linear Regression

In [ ]:
# ============================================================
# TABEL PERBANDINGAN PERFORMA
# ============================================================

print('=' * 70)
print('       PERBANDINGAN PERFORMA: MEAN BASELINE vs LINEAR REGRESSION')
print('=' * 70)
print()
print(f'  {"Metrik":<15s} | {"Mean Baseline":>18s} | {"Linear Regression":>18s} | {"Improvement":>12s}')
print(f'  {"-"*15}-+-{"-"*18}-+-{"-"*18}-+-{"-"*12}')

# MAE
imp_mae = (1 - metrics_lr['MAE'] / metrics_baseline['MAE']) * 100
print(f'  {"MAE":<15s} | ${metrics_baseline["MAE"]:>16,.2f} | ${metrics_lr["MAE"]:>16,.2f} | {imp_mae:>+10.1f}%')

# MSE
imp_mse = (1 - metrics_lr['MSE'] / metrics_baseline['MSE']) * 100
print(f'  {"MSE":<15s} | {metrics_baseline["MSE"]:>17,.0f} | {metrics_lr["MSE"]:>17,.0f} | {imp_mse:>+10.1f}%')

# RMSE
imp_rmse = (1 - metrics_lr['RMSE'] / metrics_baseline['RMSE']) * 100
print(f'  {"RMSE":<15s} | ${metrics_baseline["RMSE"]:>16,.2f} | ${metrics_lr["RMSE"]:>16,.2f} | {imp_rmse:>+10.1f}%')

# R² Score
print(f'  {"R² Score":<15s} | {metrics_baseline["R² Score"]:>18.4f} | {metrics_lr["R² Score"]:>18.4f} | {"N/A":>12s}')

print()
print('=' * 70)
print()

# Ringkasan
print('📊 RINGKASAN:')
print(f'   • Linear Regression mengurangi error (MAE) sebesar {abs(imp_mae):.1f}% dibanding Baseline.')
print(f'   • Model menjelaskan {metrics_lr["R² Score"]*100:.1f}% variasi harga rumah (R² = {metrics_lr["R² Score"]:.4f}).')
print(f'   • Rata-rata kesalahan prediksi: ${metrics_lr["MAE"]:,.0f} per rumah.')

In [ ]:
# Tabel perbandingan dalam DataFrame (untuk lampiran laporan)
df_comparison = pd.DataFrame([metrics_baseline, metrics_lr])
df_comparison = df_comparison.set_index('Model')
df_comparison['MAE'] = df_comparison['MAE'].map('${:,.2f}'.format)
df_comparison['MSE'] = df_comparison['MSE'].map('{:,.0f}'.format)
df_comparison['RMSE'] = df_comparison['RMSE'].map('${:,.2f}'.format)
df_comparison['R² Score'] = df_comparison['R² Score'].map('{:.4f}'.format)

print('📋 Tabel Perbandingan (format DataFrame):')
print()
df_comparison

---
## 7. Simpan Model ke File `housing_model.pkl`

In [ ]:
# Simpan model menggunakan joblib
model_filename = 'housing_model.pkl'
joblib.dump(model, model_filename)

print(f'💾 Model berhasil disimpan ke "{model_filename}"')
print(f'\n   Untuk memuat kembali model:')
print(f'   >>> model = joblib.load("{model_filename}")')

In [ ]:
# Verifikasi: muat ulang model dan uji prediksi
model_loaded = joblib.load(model_filename)

# Tes prediksi dengan data sampel
sample = X_test.iloc[:3]
pred_original = model.predict(sample)
pred_loaded = model_loaded.predict(sample)

print('🔍 Verifikasi Model yang Disimpan:')
print(f'   Prediksi model asli   : {[f"${p:,.0f}" for p in pred_original]}')
print(f'   Prediksi model loaded : {[f"${p:,.0f}" for p in pred_loaded]}')
print(f'   Hasil identik         : {np.allclose(pred_original, pred_loaded)} ✅')

---
## ✅ Ringkasan Pipeline

| Tahap | Keterangan |
|-------|------------|
| **Input** | `train_clean.csv` (2,793 baris × 6 kolom) |
| **Fitur** | Gr Liv Area, Overall Qual, Garage Cars, Total Bsmt SF, Year Built |
| **Target** | SalePrice |
| **Split** | 80% Training, 20% Testing (random_state=42) |
| **Baseline** | Mean Baseline (rata-rata SalePrice training) |
| **Model** | Linear Regression (Scikit-learn) |
| **Evaluasi** | MAE, MSE, RMSE, R² Score |
| **Output** | `housing_model.pkl` (model tersimpan via joblib) |